In [2]:
import phdthesisplotstyle as phd
import matplotlib.pyplot as plt
import h5py
import numpy as np
from tol_colors import tol_cset, tol_cmap
tolc = tol_cset("bright")
plt.style.use(phd.PHDTHESISPLOTSTYLE)
import glob
import os
from tqdm import tqdm
import awkward as ak

import h5py
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from glob import glob
import os
import json

In [3]:
import matplotlib.pyplot as plt
import json
from glob import glob
import dbetto as db

In [4]:
file_string = "runtime_estimates.json"

In [5]:
n_primaries = 2e6

In [6]:
simple_default = ["physics_list","em","Livermore"]

In [7]:
files = glob("**/" + file_string,recursive=True)

In [8]:
def generate_output(files):
    output = {}
    for file in files:
        folders = file.split("/")
        tmp_output = output
        for folder in folders:
            if folder == "gen":
                continue
            if folder == file_string:
                tmp_output["data"] = json.load(open(file))
            else: 
                if folder not in tmp_output:
                    tmp_output[folder] = {}
                tmp_output = tmp_output[folder]
    return db.AttrsDict(output)

In [9]:
output = generate_output(files)

In [10]:
def iterate_nested_dict(d, parent_key=''):
    for k, v in d.items():
        full_key = f"{parent_key}.{k}" if parent_key else k
        if isinstance(v, dict):
            yield from iterate_nested_dict(v, full_key)
        else:
            yield full_key, v

In [11]:
def generate_project_folders(output):
    project_folders = {}
    for key, value in iterate_nested_dict(output):
        if not "data" in key:
            continue
        folder_list = key.split(".")
        data_idx = folder_list.index("data")
        folder_list = folder_list[:data_idx]
        tmp_dict = project_folders
        for i in range(0,len(folder_list)):
            if i == len(folder_list) - 1:
                tmp_dict[folder_list[i]] = folder_list[-1]
            else:
                if folder_list[i] not in tmp_dict:
                    tmp_dict[folder_list[i]] = {}
                tmp_dict = tmp_dict[folder_list[i]]
    return db.AttrsDict(project_folders)

In [ ]:
def draw_project(data,project_list):
    x_labels = []
    for lab in list(data.keys()):
        if "data" in data[lab]:
            x_labels.append(lab)
    x_labels.sort()

    
    y_rate = [data[project]["data"]["event_rate"]["val"] for project in x_labels]
    y_rate_unc = [data[project]["data"]["event_rate"]["std"]/10 for project in x_labels]
    y_rate_median = [np.median(n_primaries/np.array(data[project]["data"]["raw"]["runtimes"])) for project in x_labels]

    if len(x_labels) == 1:
        x_labels.append("default")
        tmp = output["simple"][project_list[1]]
        for lab in simple_default:
            tmp = tmp[lab]
        y_rate.append(tmp["data"]["event_rate"]["val"])
        y_rate_unc.append(tmp["data"]["event_rate"]["std"])
        y_rate_median.append(np.median(n_primaries/np.array(tmp["data"]["raw"]["runtimes"])))

    fig, ax = plt.subplots(figsize=(phd.figsizes.TextFigure()[0]/6 * len(x_labels),phd.figsizes.TextFigure()[1]))

    ax.errorbar(x_labels,y_rate,yerr=y_rate_unc,fmt='.')
    ax.plot(x_labels,y_rate_median,marker='x',lw=0,color='black')
    plt.xticks(rotation=90)
    ax.set_ylabel("Event rate [evts/s]")
    ax.set_title(".".join(project_list))
    ax.set_ylim(0,1.1*max(y_rate))
    ax.set_xlim(-0.5,len(x_labels)-0.5)
    plt.savefig("./figs/" + "_".join(project_list) + ".png",bbox_inches='tight')
    plt.close()

In [26]:
def generate_overview_plots(output):
    project_folders = generate_project_folders(output)
    projects = []
    for key, value in iterate_nested_dict(project_folders):
        lists = key.split(".")
        if lists[:-1] not in projects:
            projects.append(lists[:-1])
    for project in projects:
        tmp_dict = output
        for pr in project:
            tmp_dict = tmp_dict[pr]
        draw_project(tmp_dict,project)
        

In [27]:
generate_overview_plots(output)

In [208]:
a = 2e6 / np.array(output["simple"]["electron"]["multithreaded"]["m1"]["data"]["raw"]["runtimes"])
b = 2e6 / np.array(output["simple"]["electron"]["multithreaded"]["m2"]["data"]["raw"]["runtimes"])

In [209]:
np.mean(a),np.std(a),np.mean(b),np.std(b)

(27330.594895547765, 567.7726789879872, 30085.52711515958, 662.1268996585467)

In [210]:
a = np.array(output["simple"]["electron"]["multithreaded"]["m1"]["data"]["raw"]["runtimes"])
b = np.array(output["simple"]["electron"]["multithreaded"]["m2"]["data"]["raw"]["runtimes"])

In [211]:
np.mean(a),np.std(a),np.mean(b),np.std(b)

(73.21, 1.5381482373295492, 66.51, 1.493284969454926)